In [1]:
import torch
import numpy as np
import drone_2d_custom_gym_env.drone_2d_env
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
import gym
from ppo import PPO
from utils import plot_learning_curve, Logger

pygame-ce 2.5.7 (SDL 2.32.10, Python 3.12.12)


objc[12997]: Class SDLApplication is implemented in both /Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x133fa4a68) and /Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x147d60890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[12997]: Class SDLAppDelegate is implemented in both /Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x133fa4ab8) and /Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x147d608e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[12997]: Class SDLTranslatorResponder is implemented

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
CASE_ID = 5

CASES = {
    1: {"initial_throw": True, "initial_force": 5000, "initial_rotation_force": 600,"wind":None,"wind_magnitude":100},
    2: {"initial_throw": True, "initial_force": 12000, "initial_rotation_force": 1500,"wind":None,"wind_magnitude":100},
    3: {"initial_throw": True, "initial_force": 5000, "initial_rotation_force": 600,"wind": "Uniform","wind_magnitude":100},
    4: {"initial_throw": True, "initial_force": 12000, "initial_rotation_force": 1500,"wind": "Uniform","wind_magnitude":100},
    5: {"initial_throw": True, "initial_force": 5000, "initial_rotation_force": 600,"wind": "Random","wind_magnitude":100},
    6: {"initial_throw": True, "initial_force": 12000, "initial_rotation_force": 1500,"wind": "Random","wind_magnitude":100}
}

def make_env(case_id, render_sim = False):
    case = CASES[case_id]
    return gym.make(
        "drone-2d-custom-v0",
        render_sim=render_sim,
        render_path=True,
        render_shade=True,
        shade_distance=70,
        n_steps=500,
        n_fall_steps=5,
        change_target=True,
        initial_throw=case["initial_throw"],
        initial_force=case["initial_force"],
        initial_rotation_force=case["initial_rotation_force"],
    )

NB_EPISODES = 100
EVAL_EP_FREQ = 50

training_env = make_env(CASE_ID)
eval_env = make_env(CASE_ID)

ppo_agent = PPO(
    training_env = training_env,
    eval_env = eval_env,
    policy_neurons_size = [128, 128],
    value_neurons_size = [128, 128],
    lr_policy = 3e-4,
    lr_value = 3e-4,
    gamma = 0.99,
    time_steps_per_batch = 2048,
    epochs = 3,
    epsilon = 0.2,
    mini_batch_size = 64,
    nb_eval_episodes = 100,
    logger = Logger.PPO(CASE_ID)
)

eval_returns = []
current_best = -np.inf

for ep in range(NB_EPISODES):
    
    ppo_agent.learn()
    
    if ep % EVAL_EP_FREQ == 0:
        avg_return, std_return = ppo_agent.evaluation_phase()
        eval_returns.append(avg_return)

        if avg_return > current_best:
            current_best = avg_return

        print(f'ep: {ep}, avg eval return {round(avg_return, 2)}, best avg eval return: {round(current_best, 2)}')

ppo_agent.logger.log_training_metrics()

/Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:187: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initialisation to be passed additional information.
  logger.warn(
/Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 

ep: 0, avg eval return -100.1, best avg eval return: -100.1
